# InstaNovo de novo predictions — Ecoli_EV_2 with FINETUNED model

Mirrors `instanovo_colab_ecoli.ipynb` but uses the fine-tuned
checkpoint (`model_finetune/instanovo/model_best.ckpt`, output of
`instanovo_colab_finetune.ipynb`) and runs only on **Ecoli_EV_2** (the
held-out test fraction; Ecoli_EV_1 was used for fine-tuning).

Outputs three CSVs for the Jetson-side FDR pipeline (dir naming matches
`run_conversions_finetune.sh` and the existing Casanovo finetune layout —
`_finetune` comes BEFORE `_mgf` / `_mgf_decoy`):
- `result_finetune/instanovo/ecoli/Ecoli_EV_2.csv`             (mzML)
- `result_finetune_mgf/instanovo/ecoli/Ecoli_EV_2.csv`         (MGF)
- `result_finetune_mgf_decoy/instanovo/ecoli/Ecoli_EV_2.decoy.csv` (decoy MGF)

InstaNovo+ (diffusion) cells are intentionally omitted here — they'll
be added in a separate notebook once InstaNovo+ is fine-tuned.


In [1]:
!nvidia-smi

Sat May  2 02:48:55 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P0             50W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## Install dependencies

In [2]:
try:
  import instanovo
except ImportError:
  !pip install "instanovo[cu126]>=1.2.2" pyopenms-viz
  print('Installation complete. Restarting runtime to apply changes...')
  import os
  os.kill(os.getpid(), 9)

## Sync inputs from Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p /content/data/ecoli
!mkdir -p /content/data_mgf/ecoli
!mkdir -p /content/data_mgf_decoy/ecoli
!mkdir -p /content/model_finetune/instanovo
!mkdir -p /content/result_finetune/instanovo/ecoli
!mkdir -p /content/result_finetune_mgf/instanovo/ecoli
!mkdir -p /content/result_finetune_mgf_decoy/instanovo/ecoli

!mkdir -p /content/drive/MyDrive/DL-Project/result_finetune/instanovo/ecoli
!mkdir -p /content/drive/MyDrive/DL-Project/result_finetune_mgf/instanovo/ecoli
!mkdir -p /content/drive/MyDrive/DL-Project/result_finetune_mgf_decoy/instanovo/ecoli

# Inputs — Ecoli_EV_2 only (held-out test)
!cp /content/drive/MyDrive/DL-Project/data/ecoli/Ecoli_EV_2.mzML            /content/data/ecoli/
!cp /content/drive/MyDrive/DL-Project/data_mgf/ecoli/Ecoli_EV_2.mgf         /content/data_mgf/ecoli/
!cp /content/drive/MyDrive/DL-Project/data_mgf_decoy/ecoli/Ecoli_EV_2.decoy.mgf /content/data_mgf_decoy/ecoli/

# Fine-tuned ckpt
!cp /content/drive/MyDrive/DL-Project/model_finetune/instanovo/model_best.ckpt /content/model_finetune/instanovo/

!ls -lh /content/data/ecoli/
!ls -lh /content/data_mgf/ecoli/
!ls -lh /content/data_mgf_decoy/ecoli/
!ls -lh /content/model_finetune/instanovo/


E Coli Instanovo

In [4]:
import os

input_path = "/content/data/ecoli"
output_path = "/content/result_finetune/instanovo/ecoli"
model_path = "/content/model_finetune/instanovo/model_best.ckpt"
samples = ["Ecoli_EV_2"]

for sample in samples:
    in_file = f"{input_path}/{sample}.mzML"
    out_file = f"{output_path}/{sample}.csv"
    if os.path.exists(out_file):
        print(f"Skipping {sample} (already exists)")
        continue
    !instanovo transformer predict --data-path {in_file} --output-path {out_file} --instanovo-model {model_path} num_workers=4 batch_size=512

!cp -r /content/result_finetune/instanovo/ecoli/. /content/drive/MyDrive/DL-Project/result_finetune/instanovo/ecoli

[05/02/26 02:49:42] INFO     Initializing InstaNovo inference.                                                                                                                 
[05/02/26 02:49:44] INFO     NumExpr defaulting to 12 threads.                                                                                                                 
2026-05-02 02:49:47.690341: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-02 02:49:47.760336: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
DEBUG:2026-05-02 02:49

E Coli MGF Instanovo

In [ ]:
import os

input_path = "/content/data_mgf/ecoli"
output_path = "/content/result_finetune_mgf/instanovo/ecoli"
model_path = "/content/model_finetune/instanovo/model_best.ckpt"
samples = ["Ecoli_EV_2"]

for sample in samples:
    in_file = f"{input_path}/{sample}.mgf"
    out_file = f"{output_path}/{sample}.csv"
    if os.path.exists(out_file):
        print(f"Skipping {sample} (already exists)")
        continue
    !instanovo transformer predict --data-path {in_file} --output-path {out_file} --instanovo-model {model_path} num_workers=4 batch_size=512

!cp -r /content/result_finetune_mgf/instanovo/ecoli/. /content/drive/MyDrive/DL-Project/result_finetune_mgf/instanovo/ecoli


E Coli MGF Decoy Instanovo

In [ ]:
import os

input_path = "/content/data_mgf_decoy/ecoli"
output_path = "/content/result_finetune_mgf_decoy/instanovo/ecoli"
model_path = "/content/model_finetune/instanovo/model_best.ckpt"
samples = ["Ecoli_EV_2"]

for sample in samples:
    in_file = f"{input_path}/{sample}.decoy.mgf"
    out_file = f"{output_path}/{sample}.decoy.csv"
    if os.path.exists(out_file):
        print(f"Skipping {sample} (already exists)")
        continue
    !instanovo transformer predict --data-path {in_file} --output-path {out_file} --instanovo-model {model_path} num_workers=4 batch_size=512

!cp -r /content/result_finetune_mgf_decoy/instanovo/ecoli/. /content/drive/MyDrive/DL-Project/result_finetune_mgf_decoy/instanovo/ecoli
